# Dasar Machine Learning

## Course 3 Bagian b: K-Nearest Neighbor

Latihan ini memakai data churn pelanggan telekomunikasi. Tujuannya memprediksi apakah seorang pelanggan berhenti berlangganan berdasarkan profil dan pemakaian layanan.

### Tujuan

- Mengenali jenis variabel: biner, kategorik, ordinal, dan numerik.
- Menyandikan serta menstandarkan fitur sebelum pemodelan.
- Melatih KNN dengan beberapa nilai `k` dan memilih yang terbaik.


## Menyiapkan pustaka


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Soal 1: Memuat data

Data diambil dari alamat publik. Sebagian baris dipakai agar latihan tetap ringkas.


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/churndata_processed.csv"
)
SAMPLE_SIZE = 3000

churn_data = pd.read_csv(DATA_URL).sample(
    n=SAMPLE_SIZE, random_state=42
).reset_index(drop=True)
round(churn_data.describe(), 2)


## Soal 2: Mengelompokkan variabel

Jumlah nilai unik tiap kolom dipakai untuk menentukan jenis variabel. Kolom dengan dua nilai unik dianggap biner, kolom dengan tiga sampai enam nilai unik dianggap kategorik, sedangkan sisanya numerik atau ordinal.


In [ ]:
unique_counts = pd.DataFrame(
    [[name, churn_data[name].nunique()] for name in churn_data.columns],
    columns=["Variable", "Unique Values"],
).set_index("Variable")
unique_counts


In [ ]:
binary_variables = list(unique_counts[unique_counts["Unique Values"] == 2].index)
binary_variables


In [ ]:
categorical_variables = list(
    unique_counts[(unique_counts["Unique Values"] <= 6)
                  & (unique_counts["Unique Values"] > 2)].index
)
categorical_variables


In [ ]:
{categorical: list(churn_data[categorical].unique()) for categorical in categorical_variables}


Kolom `contract`, `satisfaction`, dan `months` bersifat ordinal karena tingkatannya punya urutan.


In [ ]:
ordinal_variables = ["contract", "satisfaction"]
churn_data["months"].unique()


In [ ]:
ordinal_variables.append("months")

numeric_variables = list(
    set(churn_data.columns)
    - set(ordinal_variables)
    - set(categorical_variables)
    - set(binary_variables)
)
numeric_variables


In [ ]:
churn_data[numeric_variables].hist(figsize=(12, 6))
plt.tight_layout()
plt.show()


Nilai `months` dikelompokkan ke lima rentang agar skalanya sebanding dengan fitur lain.


In [ ]:
churn_data["months"] = pd.cut(churn_data["months"], bins=5)
churn_data["months"].value_counts().sort_index()


## Soal 3: Penyandian dan penstandaran

KNN menghitung jarak, sehingga semua fitur harus berskala sama. Variabel ordinal disandikan dengan `LabelEncoder`, variabel biner dengan `LabelBinarizer`, dan variabel kategorik diubah menjadi kolom dummy.


In [ ]:
from sklearn.preprocessing import LabelBinarizer, LabelEncoder, MinMaxScaler

ordinal_encoder = LabelEncoder()
binary_encoder = LabelBinarizer()

for column in ordinal_variables:
    churn_data[column] = ordinal_encoder.fit_transform(churn_data[column])

for column in binary_variables:
    churn_data[column] = binary_encoder.fit_transform(churn_data[column])

categorical_variables = list(set(categorical_variables) - set(ordinal_variables))
churn_data = pd.get_dummies(
    churn_data, columns=categorical_variables, drop_first=True
).astype(float)
round(churn_data.describe().T, 3)


In [ ]:
scaler = MinMaxScaler()
churn_data[ordinal_variables + numeric_variables] = scaler.fit_transform(
    churn_data[ordinal_variables + numeric_variables]
)
round(churn_data.describe().T, 3)


## Soal 4: Model KNN dengan k = 3

Fitur dan target dipisahkan, lalu data dibagi 60 persen latih dan 40 persen uji.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

features = churn_data.drop(columns="churn_value")
target = churn_data["churn_value"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.4, random_state=42
)


In [ ]:
nearest_model = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
predictions = nearest_model.predict(X_test)

print(classification_report(y_test, predictions))
print("Accuracy score:", round(accuracy_score(y_test, predictions), 2))
print("F1 Score:", round(f1_score(y_test, predictions), 2))


In [ ]:
def plot_confusion(predicted, actual, title):
    figure, axis = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        confusion_matrix(actual, predicted),
        annot=True,
        fmt="d",
        ax=axis,
        xticklabels=["False", "True"],
        yticklabels=["True", "False"],
    )
    axis.set(xlabel="Ground Truth", ylabel="Prediction", title=title)
    plt.tight_layout()
    plt.show()


plot_confusion(predictions, y_test, "KNN k=3")


## Soal 5: Model KNN dengan k = 5 dan bobot jarak

Bobot jarak membuat tetangga yang lebih dekat memberi pengaruh lebih besar.


In [ ]:
weighted_model = KNeighborsClassifier(n_neighbors=5, weights="distance")
weighted_model.fit(X_train, y_train)
predictions = weighted_model.predict(X_test)

print(classification_report(y_test, predictions))
print("Accuracy score:", round(accuracy_score(y_test, predictions), 2))
print("F1 Score:", round(f1_score(y_test, predictions), 2))


In [ ]:
plot_confusion(predictions, y_test, "KNN k=5 distance weighted")


## Soal 6: Mencari nilai k terbaik

Nilai `k` diuji dari 1 sampai 39. Dua ukuran dipakai: F1 Score dan tingkat kesalahan, yaitu satu dikurangi accuracy.


In [ ]:
max_k = 40
scores = []

for k in range(1, max_k):
    model = KNeighborsClassifier(n_neighbors=k, weights="distance")
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    scores.append(
        {
            "K": k,
            "F1 Score": f1_score(y_test, predicted),
            "Error Rate": 1 - accuracy_score(y_test, predicted),
        }
    )

score_table = pd.DataFrame(scores).set_index("K")
score_table.head()


In [ ]:
sns.set_context("talk")
sns.set_style("ticks")

figure, axes = plt.subplots(nrows=2, figsize=(10, 10))
score_table["F1 Score"].plot(ax=axes[0], linewidth=4, marker="o")
axes[0].set(xlabel="K", ylabel="F1 Score", title="KNN F1 Score")
score_table["Error Rate"].plot(ax=axes[1], linewidth=4, marker="o")
axes[1].set(xlabel="K", ylabel="Error Rate", title="KNN Elbow Curve")
plt.tight_layout()
plt.show()


## Ringkasan

Nilai `k` kecil membuat model peka terhadap derau, sedangkan `k` besar memperhalus batas keputusan. Kurva elbow membantu memilih `k` yang memberi kesalahan paling rendah tanpa kehilangan daya pisah.
